# Vote and Verify - Detect-then-Verify Pipeline

- **STEP 1+2:** 4 BERT models independently detect wrong words and vote.
- **STEP 3:** FLAN-T5 verifies each flagged word (yes/no) - it does NOT rewrite the paragraph.

**Run Cell 1 ONCE** to load all 5 models. Then re-run the detection cells as many times as you like without reloading - that's what makes it fast.

## Cell 1 - Imports + load ALL models (run ONCE)

In [ ]:
pip install pyspellchecker ipykernel openpyxl

In [2]:
import pandas as pd
import json
import re
import torch

from spellchecker import SpellChecker
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    AutoModelForSeq2SeqLM,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

BERT_MODELS = {
    "BioClinicalBERT": "emilyalsentzer/Bio_ClinicalBERT",
    "PubMedBERT":      "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
    "ClinicalBERT":    "medicalai/ClinicalBERT",
    "SapBERT":         "cambridgeltl/SapBERT-from-PubMedBERT-fulltext",
}
CORRECTION_MODEL_NAME = "google/flan-t5-base"

spell = SpellChecker()

bert = {}
for label, name in BERT_MODELS.items():
    print(f"Loading detector {label} ...")
    tok = AutoTokenizer.from_pretrained(name)
    mdl = AutoModelForMaskedLM.from_pretrained(name).to(device)
    mdl.eval()
    bert[label] = (tok, mdl)

print(f"Loading verifier FLAN-T5 ...")
t5_tokenizer = AutoTokenizer.from_pretrained(CORRECTION_MODEL_NAME)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(CORRECTION_MODEL_NAME).to(device)
t5_model.eval()

print(f"\nAll models loaded on {device}. You can now run the detection cells repeatedly.")

Loading detector BioClinicalBERT ...
Loading detector PubMedBERT ...


Some weights of the model checkpoint at microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Loading detector ClinicalBERT ...
Loading detector SapBERT ...


Some weights of BertForMaskedLM were not initialized from the model checkpoint at cambridgeltl/SapBERT-from-PubMedBERT-fulltext and are newly initialized: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading verifier FLAN-T5 ...

All models loaded on cpu. You can now run the detection cells repeatedly.


## Cell 2 - Config + spell-check gate (run once; re-run if you edit)

In [3]:
SUSPICION_THRESHOLD = 0.001   # a BERT model flags a non-word below this probability
MIN_VOTES = 2                 # word reported only if >= this many models flag it
MIN_WORD_LENGTH = 0
HL_OPEN, HL_CLOSE = "<<", ">>"


def is_real_word(clean_word):
    """True if the spell-checker recognises this as a real word.
    Real words (antibody, dengue, saline) are skipped before the
    expensive models run."""
    return len(spell.unknown([clean_word.lower()])) == 0

print("Config ready.")

Config ready.


## Cell 3 - Detection functions (STEP 1+2: BERT detect + vote)

In [4]:
def word_probability(tokenizer, model, plain_words, index, clean_word):
    """Mask the word (correct number of [MASK] tokens) in the full
    paragraph; return the model's average probability for it."""
    word_token_ids = tokenizer(clean_word, add_special_tokens=False)["input_ids"]
    n_tokens = len(word_token_ids)
    if n_tokens == 0:
        return None

    masked = plain_words.copy()
    masked[index] = " ".join([tokenizer.mask_token] * n_tokens)
    masked_sentence = " ".join(masked)

    inputs = tokenizer(masked_sentence, return_tensors="pt", truncation=True, max_length=512).to(device)
    mask_positions = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
    if len(mask_positions) == 0:
        return None

    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits[0], dim=-1)

    usable = min(n_tokens, len(mask_positions))
    token_probs = [probs[mask_positions[k].item(), word_token_ids[k]].item() for k in range(usable)]
    return sum(token_probs) / len(token_probs) if token_probs else None


def detect_with_model(tokenizer, model, word_spans):
    """Return {word_index: confidence} for words this model flags."""
    plain_words = [w for (w, _s, _e) in word_spans]
    flagged = {}
    for i, (raw_word, _start, _end) in enumerate(word_spans):
        clean_word = re.sub(r"[^A-Za-z]", "", raw_word)
        if len(clean_word) < MIN_WORD_LENGTH:
            continue
        if is_real_word(clean_word):        # GATE: skip real words
            continue
        prob = word_probability(tokenizer, model, plain_words, i, clean_word)
        if prob is None:
            continue
        if prob < SUSPICION_THRESHOLD:
            flagged[i] = 1.0 - prob
    return flagged


def vote(word_spans):
    """Run all BERT models, then merge by voting."""
    tally = {}
    for label, (tok, mdl) in bert.items():
        flagged = detect_with_model(tok, mdl, word_spans)
        for idx, conf in flagged.items():
            tally.setdefault(idx, {})[label] = conf

    detections = []
    for idx, model_confs in tally.items():
        votes = len(model_confs)
        if votes < MIN_VOTES:
            continue
        raw_word, start, end = word_spans[idx]
        combined = sum(model_confs.values()) / votes
        detections.append({
            "word": raw_word, "start": start, "end": end, "votes": votes,
            "model_confidences": {k: round(v, 6) for k, v in model_confs.items()},
            "combined_confidence": round(combined, 6),
        })
    detections.sort(key=lambda d: d["start"])
    return detections


def highlight_paragraph(text, detections):
    highlighted = text
    for d in sorted(detections, key=lambda x: x["start"], reverse=True):
        s, e = d["start"], d["end"]
        highlighted = highlighted[:s] + HL_OPEN + highlighted[s:e] + HL_CLOSE + highlighted[e:]
    return highlighted

print("Detection functions ready.")

Detection functions ready.


## Cell 4 - Verification functions (STEP 3: FLAN-T5 yes/no, no rewriting)

In [5]:
def verify_word_with_flan_t5(text, word):
    """Ask FLAN-T5 whether ONE flagged word is really wrong, given the
    full paragraph. Returns (verdict, raw_answer). No rewriting."""
    prompt = f'''You are a medical language reviewer.
Read the paragraph below. A word from it has been flagged as
possibly wrong.

Decide whether the word "{word}" is actually wrong in this
paragraph - that is, misspelled, medically incorrect, or
grammatically incorrect.

Answer with only one word: "yes" if it is wrong, "no" if it is
correct.

Paragraph:
{text}

Is "{word}" wrong? Answer yes or no.
'''
    inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = t5_model.generate(**inputs, max_new_tokens=5, num_beams=4, early_stopping=True)
    answer = t5_tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()

    if "yes" in answer:
        verdict = "yes"
    elif "no" in answer:
        verdict = "no"
    else:
        verdict = "unsure"
    return verdict, answer


def verify_flagged_words(text, detections):
    """Run FLAN-T5 over every BERT-flagged word and attach its verdict."""
    verified = []
    for d in detections:
        verdict, raw = verify_word_with_flan_t5(text, d["word"])
        item = dict(d)
        item["flan_verdict"] = verdict
        item["flan_answer"] = raw
        item["really_wrong"] = (verdict == "yes")
        verified.append(item)
    return verified


def process_text(text):
    if pd.isna(text):
        return {"highlighted_text": "", "flagged_words": [], "confirmed_wrong_words": [], "combined_confidence": 0.0}
    text = str(text)
    word_spans = [(m.group(), m.start(), m.end()) for m in re.finditer(r"\S+", text)]

    detections = vote(word_spans)                       # STEP 1+2
    verified = verify_flagged_words(text, detections)   # STEP 3
    confirmed = [v["word"] for v in verified if v["really_wrong"]]
    overall = sum(v["combined_confidence"] for v in verified) / len(verified) if verified else 0.0

    return {
        "highlighted_text": highlight_paragraph(text, detections),
        # "flagged_words": verified,
        # "confirmed_wrong_words": confirmed,
        # "combined_confidence": round(overall, 6),
    }

print("Verification functions ready.")

Verification functions ready.


## Cell 5 - Quick test on one paragraph (instant, re-run freely)

In [ ]:
sample = "the patient is suspected to have tengu. dengue test cbc. abu antibody titer."

result = process_text(sample)
print(result["highlighted_text"])
print()
print(json.dumps(result, indent=2, ensure_ascii=False))

## Cell 6 - Run over the whole Excel file and save

In [ ]:
INPUT_FILE = r"C:\Users\HITESH S ROY\OneDrive\Documents\Medical_Checker\medical_checker.xlsx"
OUTPUT_FILE = r"C:\Users\HITESH S ROY\OneDrive\Documents\Medical_Checker\Output\Vote_and_Correct_Notebook.xlsx"
INPUT_COLUMN = "error"

df = pd.read_excel(INPUT_FILE)
if INPUT_COLUMN not in df.columns:
    raise Exception(f"Excel file must contain '{INPUT_COLUMN}' column")

outputs = []
for index, row in df.iterrows():
    result = process_text(row[INPUT_COLUMN])
    outputs.append(json.dumps(result, ensure_ascii=False))
    print(f"Row {index + 1}: {len(result['flagged_words'])} flagged by BERT (>= {MIN_VOTES} votes), "
          f"{len(result['confirmed_wrong_words'])} confirmed wrong by FLAN-T5")

df["vote_and_verify"] = outputs
df.to_excel(OUTPUT_FILE, index=False)
print(f"\nSaved: {OUTPUT_FILE}")